In [1]:
import pandas as pd
from mlds import data_loader
import warnings, re, os
warnings.filterwarnings('ignore')

manager = data_loader.SlotDataManager(data_folder="data/output")

def parse_spans(spans_str: str):
    """Parse spans string from format like '10:29:SL:ACCOUNT_TYPE' into list of dictionaries."""
    if not spans_str:
        return []
    
    spans = []
    for span in spans_str.split(','):
        if ':' not in span:
            continue
        span = span.strip()
        parts = span.split(':')
        if len(parts) >= 4:
            start, end, _, label = parts[:4]
            spans.append({
                "start_byte": int(start),
                "limit_byte": int(end),
                "label": label
            })
    return spans

def parse_logical_form(logical_form: str) -> str:
    """Convert logical form to target format."""
    # [IN:balance [SL:ACCOUNT_TYPE àpò ìfowópamọ́ Dorm] ] => ACCOUNT_TYPE: àpò ìfowópamọ́ Dorm
    # [IN:translate [SL:DISH_OR_FOOD nemi] [SL:LANGUAGE_NAME fɔ̃gbe] ] => DISH_OR_FOOD: nemi $$ LANGUAGE_NAME: fɔ̃gbe
    parts = []
    for part in re.findall(r"\[SL:([A-Z_]+) ([^\]]+)\]", logical_form):
        label, value = part
        parts.append(f"{label}: {value}")
    return " $$ ".join(parts)

summary_information = {}

for language in data_loader.LANGUAGES + ["eng"]:
    data = manager.load_data(language, "split")
    
    summary_information[language] = {
        "train": len(data["train"]),
        "dev": len(data["dev"]),
        "test": len(data["test"]),
    }

    for split, item in data.items():
        # print(item.keys())
        item['spans'] = item['spans'].fillna("").apply(parse_spans)
        item['target'] = item['logical_form'].apply(parse_logical_form)
        # print(item.index)
        item['example_id'] = f"{split}-" + \
            item.index.map(lambda x: f"{x[0]}-{x[1]:08d}" if isinstance(x, tuple) else f"{x:08d}")
        item.drop(columns=['logical_form'], inplace=True) # , 'split'
        os.makedirs(f"data/cleaned-dataset/{language}", exist_ok=True)
        item.to_json(f"data/cleaned-dataset/{language}/{split}.jsonl", orient='records', lines=True, force_ascii=False)
        
summary_information = pd.DataFrame(summary_information).T
summary_information


Missing balance with {'sna', 'lug'}
Missing confirm_reservation with {'sna', 'lug'}
Missing freeze_account with {'sna', 'lug'}
Missing restaurant_reservation with {'sna', 'lug'}
Missing shopping_list_update with {'sna', 'lug'}
Missing time with {'sna', 'lug'}
Missing timezone with {'sna', 'lug'}
Missing transfer with {'sna', 'lug'}
Missing translate with {'sna', 'lug'}


,train,dev,test
amh,2240,320,640
ewe,2240,320,640
hau,2240,320,640
ibo,2240,320,640
kin,2240,320,640
lin,2240,320,640
lug,2240,320,640
orm,2240,320,640
sna,2240,320,640
sot,2240,320,640
